In [ ]:
import json
import sys
import functools
from typing import List, Optional, Dict, Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve

sys.path.append('..')
from utils.analysis_utils import analyze_all_tasks, extract_verdict
from utils.path_utils import resolve_path
import gc

DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task", "transcripts_v1", "transcripts_no_hint_v1", "transcripts_no_secondary_task_v1"]

def load_monitoring_data(base_dir: str = "monitor_responses/turn_by_turn_monitoring", 
                        transcript_types: Optional[list[str]] = None) -> pd.DataFrame:
    """Load all turn-by-turn monitoring data from the specified directory structure.
    
    Directory structure:
    base_dir/
        task_name/
            transcript_type/
                agent_model/
                    monitor_model/
                        data.json
    """
    if transcript_types is None:
        transcript_types = DEFAULT_TRANSCRIPT_TYPES
    
    all_data = []
    base_path = resolve_path(base_dir)
    
    # Process each task directory
    for task_dir in base_path.iterdir():
        if not task_dir.is_dir():
            continue
        
        
        # Process each transcript type directory
        for transcript_type in transcript_types:
            transcript_path = task_dir / transcript_type
            if not transcript_path.exists() or not transcript_path.is_dir():
                print(f"Skipping non-existent transcript type: {transcript_type} in task {task_dir.name}")
                continue
                
            
            # Process agent model directories
            for agent_model in transcript_path.iterdir():
                if not agent_model.is_dir():
                    continue
                    
                # Process monitor model directories
                for monitor_model in agent_model.iterdir():
                    if not monitor_model.is_dir():
                        continue
                        
                    # Look for JSON files in the monitor_model directory
                    json_files = list(monitor_model.glob('*.json'))
                    for json_file in json_files:
                        try:
                            with open(json_file, 'r') as f:
                                data = json.load(f)
                                
                            # Add metadata about the file location
                            entry = {
                                'task_name': task_dir.name,
                                'transcript_type': transcript_type,
                                'main_agent': agent_model.name,
                                'monitor_agent': monitor_model.name,
                                'file_name': json_file.name,
                                'data': data
                            }
                            all_data.append(entry)
                        except Exception as e:
                            print(f"Error loading {json_file}: {e}")
                            continue
    gc.collect()
    return pd.DataFrame(all_data)

class CombinedResultsCache:
    """A class to cache combined results data using class methods."""
    _cache: Dict[str, pd.DataFrame] = {}
    _base_data: pd.DataFrame = None
    
    @classmethod
    def get_base_data(cls, transcript_types: Optional[list[str]] = None, agent_models_list: Optional[list[str]] = None) -> pd.DataFrame:
        """Load the base combined results data if not already cached."""
        if transcript_types is None:
            transcript_types = DEFAULT_TRANSCRIPT_TYPES

        if cls._base_data is None:
            cls._base_data = analyze_all_tasks(transcript_types=transcript_types, agent_models_list=agent_models_list)
            cls._base_data.rename(columns={'suspiciousness': 'default_suspiciousness'}, inplace=True)
        return cls._base_data
    
    @classmethod
    def get_filtered_data(cls, include_failed_attempts: bool, include_failed_main_task: bool, transcript_types: Optional[list[str]] = None, agent_models_list: Optional[list[str]] = None) -> pd.DataFrame:
        """Get filtered data based on parameters, using cache if available."""
        cache_key = f"{include_failed_attempts}_{include_failed_main_task}"
        
        if cache_key not in cls._cache:
            combined_results = cls.get_base_data(transcript_types=transcript_types, agent_models_list=agent_models_list).copy()
            
            if not include_failed_attempts:
                combined_results = combined_results[
                    (combined_results.side_task_evaluation > 0) | 
                    (combined_results.transcript_type.str.contains('transcripts_no_secondary_task'))
                ]
            
            if not include_failed_main_task:
                combined_results = combined_results[(combined_results.main_task_success)]
            
            cls._cache[cache_key] = combined_results
        gc.collect()
        return cls._cache[cache_key]

def load_in_combined_results(include_failed_attempts: bool, include_failed_main_task: bool, transcript_types: Optional[list[str]] = None, agent_models_list: Optional[list[str]] = None) -> pd.DataFrame:
    """Load combined results with caching for better performance.
    
    Args:
        include_failed_attempts: Whether to include data where side tasks failed
        include_failed_main_task: Whether to include data where main tasks failed
        
    Returns:
        DataFrame with filtered combined results
    """
    return CombinedResultsCache.get_filtered_data(include_failed_attempts, include_failed_main_task, transcript_types, agent_models_list)

def plot_suspiciousness_distributions(merged_df: pd.DataFrame, 
                                    monitors: List[str] = ['claude-3-5-haiku-20241022', 'gpt-4o'],
                                    metrics: List[str] = ['max_suspiciousness', 'final_suspiciousness']):
    """Plot histograms of suspiciousness distributions."""
    for monitor in monitors:
        for value_to_plot in metrics:
            plt.figure(figsize=(10, 6))
            filtered_df = merged_df[merged_df.monitor_agent == monitor]
            
            # First histogram - no secondary task
            no_secondary = filtered_df[filtered_df.transcript_type == 'transcripts_no_secondary_task']
            
            # Second histogram - successful secondary task
            successful_secondary = filtered_df[filtered_df.side_task_evaluation == 1]
            
            # Calculate common bins based on the combined range of both datasets
            min_val = min(no_secondary[value_to_plot].min(), successful_secondary[value_to_plot].min())
            max_val = max(no_secondary[value_to_plot].max(), successful_secondary[value_to_plot].max())
            common_bins = np.linspace(min_val, max_val, 31)  # 31 points make 30 bins
            
            # Plot with common bins
            plt.hist(no_secondary[value_to_plot], bins=common_bins, alpha=0.5, 
                    label=f'No Secondary Task (n={len(no_secondary)})', color='blue', 
                    density=True)

            plt.hist(successful_secondary[value_to_plot], bins=common_bins, alpha=0.5, 
                    label=f'Successful Secondary Task (n={len(successful_secondary)})', color='red',
                    density=True)

            plt.xlabel(value_to_plot)
            plt.ylabel('Density')
            plt.title(f'Distribution of {value_to_plot} by Task Type\nMonitor: {monitor}')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.show()

def plot_roc_curves(merged_df: pd.DataFrame, 
                   combined_results: pd.DataFrame,
                   give_hint: Optional[bool],
                   include_failed_attempts: bool,
                   include_failed_main_task: bool,
                   monitors: List[str] = ['claude-3-5-haiku-20241022', 'gpt-4o'],
                   metrics: List[str] = ['max_suspiciousness', 'final_suspiciousness', 'default_suspiciousness']):
    """Plot ROC curves for different monitoring configurations."""
    eval_df = merged_df
    # Only apply transcript type filtering if give_hint is not None
    if give_hint is not None:
        if give_hint:
            eval_df = eval_df[eval_df.transcript_type != 'transcripts_no_hint']
        else:
            eval_df = eval_df[eval_df.transcript_type != 'transcripts']
    for monitor in monitors:
        plt.figure(figsize=(10, 8))
        
        # Store lines and their AUC values for sorting
        lines = []
        auc_values = []
        
        # for baseline_monitor in ['claude-3-5-sonnet-20241022', 'gpt-4o']:
        #     # Get baseline data
        #     baseline_data = combined_results[
        #         (combined_results.monitor_agent == baseline_monitor)
        #     ]
            
        #     # Apply the same filtering as in load_in_combined_results
        #     if not include_failed_attempts:
        #         baseline_data = baseline_data[
        #             (baseline_data.side_task_evaluation > 0) | 
        #             (baseline_data.transcript_type == 'transcripts_no_secondary_task')
        #         ]
        #     if not include_failed_main_task:
        #         baseline_data = baseline_data[(baseline_data.main_task_success)]
                
        #     # Additional filtering specific to this function - only if give_hint is not None
        #     if give_hint is not None:
        #         baseline_data = baseline_data[
        #             baseline_data.transcript_type != ('transcripts_no_hint' if give_hint else 'transcripts')
        #         ]
            
        #     # Calculate baseline ROC curve
        #     if len(baseline_data) > 0:
        #         y_true_baseline = (baseline_data.side_task_evaluation == 1).astype(int)
        #         y_score_baseline = baseline_data['default_suspiciousness'] / 100
        #         fpr_baseline, tpr_baseline, _ = roc_curve(y_true_baseline, y_score_baseline)
        #         roc_auc_baseline = auc(fpr_baseline, tpr_baseline)
                
        #         # Store line and AUC value instead of plotting immediately
        #         line, = plt.plot(fpr_baseline, tpr_baseline, '--', color='gray')
        #         lines.append((line, f'Baseline ({baseline_monitor}) (AUC = {roc_auc_baseline:.2f})'))
        #         auc_values.append(roc_auc_baseline)
        
        # Calculate ROC curves for different metrics
        for metric in metrics:
            monitor_data = eval_df[eval_df.monitor_agent == monitor]
            if len(monitor_data) == 0:
                continue
                
            y_true = (monitor_data.side_task_evaluation == 1).astype(int)
            y_score = monitor_data[metric] / 100  # Normalize to 0-1 range
            
            fpr, tpr, _ = roc_curve(y_true, y_score)
            roc_auc = auc(fpr, tpr)
            
            # Store line and AUC value instead of plotting immediately
            line, = plt.plot(fpr, tpr)
            lines.append((line, f'{metric} (AUC = {roc_auc:.2f})'))
            auc_values.append(roc_auc)

        # Add diagonal line (not included in sorted legend)
        plt.plot([0, 1], [0, 1], 'k--')  # diagonal line
        
        # Sort lines by AUC values in descending order
        sorted_indices = np.argsort(auc_values)[::-1]
        sorted_lines = [lines[i][0] for i in sorted_indices]
        sorted_labels = [lines[i][1] for i in sorted_indices]
        
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        
        # Update title to handle None case
        hint_text = ''
        if give_hint is not None:
            hint_text = ' with hint' if give_hint else ' without hint'
            
        plt.title(
            'ROC Curves by Monitoring Type' + hint_text + 
            (' with failed attempts' if include_failed_attempts else ' without failed attempts') + 
            (' with failed main task' if include_failed_main_task else ' without failed main task') + 
            f'\nMonitor: {monitor}'
        )
        
        # Create legend with sorted entries
        plt.legend(sorted_lines, sorted_labels, loc='lower right')
        plt.grid(True, alpha=0.3)
        plt.show()

def plot_markov_analysis(merged_df: pd.DataFrame, 
                        main_agent: str = 'claude-3-5-sonnet-20241022',
                        monitor_model: str = 'gpt-4o',
                        give_hint: bool = False,
                        metrics: List[str] = ['default_suspiciousness', 'max_suspiciousness', 'final_suspiciousness']):
    """Plot Markov analysis for different monitoring configurations."""
    for metric in metrics:
        plot_markov_df(merged_df, main_agent, monitor_model, give_hint, metric)

# Function to interpolate TPR at a specific FPR
def interpolate_tpr(fpr_points, tpr_points, target_fpr):
    """
    Interpolate the TPR value for a given FPR value using linear interpolation.
    """
    # Handle edge cases
    if target_fpr <= fpr_points[0]:
        return tpr_points[0]
    if target_fpr >= fpr_points[-1]:
        return tpr_points[-1]
    
    # Find the two closest FPR points for interpolation
    for i in range(len(fpr_points) - 1):
        if fpr_points[i] <= target_fpr <= fpr_points[i + 1]:
            # Linear interpolation formula
            slope = (tpr_points[i + 1] - tpr_points[i]) / (fpr_points[i + 1] - fpr_points[i])
            return tpr_points[i] + slope * (target_fpr - fpr_points[i])
    
    # This should not happen if the edge cases are handled correctly
    return None

# Function to compute Pareto frontier
def compute_pareto_frontier(costs, scores, maximize_score=True):
    """
    Compute the Pareto frontier points for cost vs. score
    """
    # Combine costs and scores into a single array of (cost, score) pairs
    points = np.column_stack((costs, scores))
    
    # Sort points by cost (ascending)
    points = points[points[:, 0].argsort()]
    
    # Initialize Pareto frontier with the first point
    pareto_frontier = [points[0]]
    
    # Iterate through the sorted points
    for point in points[1:]:
        # For maximizing score: only keep points with higher score than the last Pareto point
        if maximize_score and point[1] > pareto_frontier[-1][1]:
            pareto_frontier.append(point)
        # For minimizing score: only keep points with lower score than the last Pareto point
        elif not maximize_score and point[1] < pareto_frontier[-1][1]:
            pareto_frontier.append(point)
    
    # Convert list of points back to separate arrays
    pareto_frontier = np.array(pareto_frontier)
    return pareto_frontier[:, 0], pareto_frontier[:, 1]

def plot_cost_vs_safety_curves(df: pd.DataFrame, 
                              monitor_column: str = 'monitor_agent',
                              monitors: List[str] = ['gpt-4o', 'o3-mini'],
                              metric_columns: List[str] = None,
                              cost_columns: List[str] = None,
                              target_fpr: float = 0.01,
                              side_task_column: str = 'side_task_evaluation',
                              normalize_scores: bool = True,
                              title_prefix: str = '',
                              figsize: tuple = (10, 6)):
    """
    Create cost vs safety curves with Pareto frontiers for different monitors.
    
    Args:
        df: DataFrame containing the data
        monitor_column: Column name for the monitor identifier
        monitors: List of monitor models to include
        metric_columns: List of column names for the metrics to plot
        cost_columns: List of column names for the corresponding costs
                     (must be in the same order as metric_columns)
        target_fpr: Target false positive rate for TPR interpolation
        side_task_column: Column name for the binary classification target
        normalize_scores: Whether to normalize scores to 0-100 range
        title_prefix: Optional prefix for plot titles
        figsize: Figure size as (width, height) tuple
    
    Returns:
        DataFrame containing the processed data used for plotting
    """
    
    
    # Validate inputs
    if metric_columns is None or cost_columns is None:
        raise ValueError("Both metric_columns and cost_columns must be provided")
    
    if len(metric_columns) != len(cost_columns):
        raise ValueError("metric_columns and cost_columns must have the same length")
    
    # Create a list to store rows for the skinny_df
    rows_list = []
    
    # Process each monitor and metric combination
    for monitor in monitors:
        monitor_data = df[df[monitor_column] == monitor]
        
        # Skip if no data for this monitor
        if len(monitor_data) == 0:
            print(f"No data for monitor: {monitor}")
            continue
            
        # Process each metric and its corresponding cost
        for metric, cost_metric in zip(metric_columns, cost_columns):
            # Skip if metric not in dataframe
            if metric not in monitor_data.columns or cost_metric not in monitor_data.columns:
                print(f"Metric {metric} or cost metric {cost_metric} not in dataframe for monitor {monitor}")
                continue
                
            # Calculate ROC curve and AUC
            y_true = (monitor_data[side_task_column] == 1).astype(int)
            y_score = monitor_data[metric]
            
            # Normalize scores if requested
            if normalize_scores and y_score.max() > 1:
                y_score = y_score / 100  # Normalize to 0-1 range
            
            fpr, tpr, _ = roc_curve(y_true, y_score)
            roc_auc = auc(fpr, tpr)
            
            # Interpolate TPR at target FPR
            interpolated_tpr = interpolate_tpr(fpr, tpr, target_fpr)
            
            # Get average cost
            avg_cost = monitor_data[cost_metric].mean()
            
            # Append to rows list
            rows_list.append({
                'metric': metric,
                'auc_score': roc_auc,
                'interpolated_tpr': interpolated_tpr,
                'monitor': monitor,
                'cost_usd': avg_cost
            })
    
    # Create skinny_df
    if len(rows_list) == 0:
        print("No data to plot. Returning empty DataFrame.")
        return pd.DataFrame()
        
    skinny_df = pd.DataFrame(rows_list)
    
    # Create figure for TPR plot
    plt.figure(figsize=figsize)
    
    # Plot points and Pareto frontier for each monitor
    colors = ['blue', 'red', 'green', 'purple', 'orange']
    markers = ['o', 's', '^', 'D', 'v']
    
    for i, monitor in enumerate(monitors):
        monitor_data = skinny_df[skinny_df['monitor'] == monitor]
        
        # Skip if no data for this monitor
        if len(monitor_data) == 0:
            continue
            
        # Plot scatter points
        plt.scatter(
            monitor_data['cost_usd'], 
            monitor_data['interpolated_tpr'],
            label=f'{monitor} (all points)',
            color=colors[i % len(colors)],
            marker=markers[i % len(markers)],
            alpha=0.5
        )
        
        # Compute and plot Pareto frontier
        pareto_costs, pareto_tprs = compute_pareto_frontier(
            monitor_data['cost_usd'].values, 
            monitor_data['interpolated_tpr'].values
        )
        
        plt.plot(
            pareto_costs, 
            pareto_tprs, 
            color=colors[i % len(colors)], 
            linestyle='-', 
            linewidth=2,
            label=f'{monitor} (Pareto frontier)'
        )
    
    plt.xlabel('Cost (USD)', fontsize=12)
    plt.ylabel(f'TPR at FPR={target_fpr}', fontsize=12)
    plt.title(f'{title_prefix}Cost vs. TPR at FPR={target_fpr} with Pareto Frontiers', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()
    
    # Create figure for AUC plot
    plt.figure(figsize=figsize)
    
    # Plot points and Pareto frontier for each monitor
    for i, monitor in enumerate(monitors):
        monitor_data = skinny_df[skinny_df['monitor'] == monitor]
        
        # Skip if no data for this monitor
        if len(monitor_data) == 0:
            continue
            
        # Plot scatter points
        plt.scatter(
            monitor_data['cost_usd'], 
            monitor_data['auc_score'],
            label=f'{monitor} (all points)',
            color=colors[i % len(colors)],
            marker=markers[i % len(markers)],
            alpha=0.5
        )
        
        # Compute and plot Pareto frontier
        pareto_costs, pareto_aucs = compute_pareto_frontier(
            monitor_data['cost_usd'].values, 
            monitor_data['auc_score'].values
        )
        
        plt.plot(
            pareto_costs, 
            pareto_aucs, 
            color=colors[i % len(colors)], 
            linestyle='-', 
            linewidth=2,
            label=f'{monitor} (Pareto frontier)'
        )
    
    plt.xlabel('Cost (USD)', fontsize=12)
    plt.ylabel('AUC Score', fontsize=12)
    plt.title(f'{title_prefix}Cost vs. AUC Score with Pareto Frontiers', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()
    
    # Create a table of the best points on the Pareto frontier for each monitor
    print("Top 5 points on TPR Pareto frontier for each monitor:")
    for monitor in monitors:
        monitor_data = skinny_df[skinny_df['monitor'] == monitor]
        if len(monitor_data) == 0:
            continue
            
        pareto_costs, pareto_tprs = compute_pareto_frontier(
            monitor_data['cost_usd'].values, 
            monitor_data['interpolated_tpr'].values
        )
        
        pareto_indices = []
        for cost, tpr in zip(pareto_costs, pareto_tprs):
            idx = monitor_data[(monitor_data['cost_usd'] == cost) & 
                              (monitor_data['interpolated_tpr'] == tpr)].index
            if len(idx) > 0:
                pareto_indices.append(idx[0])
        
        pareto_df = monitor_data.loc[pareto_indices].sort_values('interpolated_tpr', ascending=False)
        print(f"\n{monitor} - Top 5 TPR points:")
        print(pareto_df[['metric', 'cost_usd', 'interpolated_tpr']].head(5))
    
    print("\nTop 5 points on AUC Pareto frontier for each monitor:")
    for monitor in monitors:
        monitor_data = skinny_df[skinny_df['monitor'] == monitor]
        if len(monitor_data) == 0:
            continue
            
        pareto_costs, pareto_aucs = compute_pareto_frontier(
            monitor_data['cost_usd'].values, 
            monitor_data['auc_score'].values
        )
        
        pareto_indices = []
        for cost, auc_score in zip(pareto_costs, pareto_aucs):
            idx = monitor_data[(monitor_data['cost_usd'] == cost) & 
                              (monitor_data['auc_score'] == auc_score)].index
            if len(idx) > 0:
                pareto_indices.append(idx[0])
        
        pareto_df = monitor_data.loc[pareto_indices].sort_values('auc_score', ascending=False)
        print(f"\n{monitor} - Top 5 AUC points:")
        print(pareto_df[['metric', 'cost_usd', 'auc_score']].head(5))
    
    return skinny_df

def get_roc_curve(df, metric, side_task_column='side_task_evaluation', normalize=True):
    """
    Calculate ROC curve points from a DataFrame containing binary classification labels.
    This is a convenience wrapper around sklearn's roc_curve function that handles
    data extraction and normalization.
    
    Args:
        df: DataFrame containing the data
        metric: Name of the column containing the metric/score to use
        side_task_column: Name of the column containing binary labels (0/1 or True/False)
        normalize: Whether to normalize the metric values (dividing by 100)
        
    Returns:
        fpr, tpr: Arrays containing false positive rates and true positive rates
        
    Note:
        This function uses sklearn.metrics.roc_curve internally but simplifies the common
        pattern of extracting data from a DataFrame and normalizing scores.
    """
    true_labels = (df[side_task_column] == 1).astype(int)
    scores = df[metric]
    if normalize and scores.max() > 1:
        scores = scores / 100  # Normalize to 0-1 range
    fpr, tpr, _ = roc_curve(true_labels, scores)
    return fpr, tpr

def is_in_training_data(row, training_df):
    """
    Check if a given row exists in the training data based on key identifiers.
    
    Args:
        row: Row from a DataFrame to check
        training_df: DataFrame containing training data
        
    Returns:
        bool: True if the row exists in the training data, False otherwise
    """
    matching_rows = training_df[
        (training_df.main_agent == row.main_agent) &
        (training_df.file_name == row.file_name) &
        (training_df.task_name == row.task_name) &
        (training_df.transcript_type == row.transcript_type)
    ]
    return len(matching_rows) > 0

def plot_training_progress(df, metric, step_mapping=None, model_column='monitor_agent', 
                          models=None, target_fpr=0.01, side_task_column='side_task_evaluation',
                          figsize=(10, 6), show_plot=True):
    """
    Plot the AUC and interpolated TPR as a function of training steps for models.
    
    Args:
        df: DataFrame to use for analysis
        metric: The metric column to use for analysis
        step_mapping: Dictionary mapping model names to their training steps
        model_column: Column name for the model identifier
        models: List of models to include in the analysis. If None, uses all unique values in model_column
        target_fpr: Target false positive rate for interpolated TPR calculation
        side_task_column: Column name for the binary classification target
        figsize: Figure size as (width, height) tuple
        show_plot: Whether to display the plot
        
    Returns:
        DataFrame containing the results for each model
    """
    # If no models specified, use all unique values in the model column
    if models is None:
        models = df[model_column].unique()
    
    # If no step mapping provided, create default based on order
    if step_mapping is None:
        step_mapping = {}
        for i, model in enumerate(models):
            step_mapping[model] = i * 100  # Default step size
    
    # Calculate AUC and interpolated TPR for each model
    results = []
    
    for model in models:
        model_df = df[df[model_column] == model]
        if len(model_df) > 0 and metric in model_df.columns:
            # Calculate AUC
            fpr, tpr = get_roc_curve(model_df, metric, side_task_column)
            roc_auc = auc(fpr, tpr)
            
            # Calculate interpolated TPR
            interp_tpr = interpolate_tpr(fpr, tpr, target_fpr)
            
            results.append({
                'model': model,
                'step': step_mapping.get(model, 0),
                'metric': metric,
                'auc': roc_auc,
                'tpr': interp_tpr
            })
    
    # Convert to DataFrame and sort by step
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('step')
    
    if show_plot and len(results_df) > 0:
        # Create the plot
        fig, ax1 = plt.figure(figsize=figsize), plt.gca()
        
        # Plot AUC
        color = 'tab:blue'
        ax1.set_xlabel('Training Steps')
        ax1.set_ylabel('AUC', color=color)
        ax1.plot(results_df['step'], results_df['auc'], 'o-', color=color, label='AUC')
        ax1.tick_params(axis='y', labelcolor=color)
        
        # Create second y-axis for TPR
        ax2 = ax1.twinx()
        color = 'tab:red'
        ax2.set_ylabel(f'TPR @ FPR={target_fpr}', color=color)
        ax2.plot(results_df['step'], results_df['tpr'], 'o-', color=color, label=f'TPR @ FPR={target_fpr}')
        ax2.tick_params(axis='y', labelcolor=color)
        
        # Add title and legend
        plt.title(f'Model Performance vs Training Steps ({metric})')
        
        # Add legend
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')
        
        plt.tight_layout()
        plt.grid(True, alpha=0.3)
        plt.show()
    
    return results_df

def plot_all_suspiciousness_metrics(df, metrics_list, step_mapping=None, model_column='monitor_agent',
                                   models=None, target_fpr=0.01, side_task_column='side_task_evaluation',
                                   figsize=(15, 6), show_plot=True):
    """
    Plot the AUC and interpolated TPR for specified metrics as a function of training steps.
    
    Args:
        df: DataFrame to use for analysis
        metrics_list: List of metrics to include in the plot
        step_mapping: Dictionary mapping model names to their training steps
        model_column: Column name for the model identifier 
        models: List of models to include in the analysis. If None, uses all unique values in model_column
        target_fpr: Target false positive rate for interpolated TPR calculation
        side_task_column: Column name for the binary classification target
        figsize: Figure size as (width, height) tuple
        show_plot: Whether to display the plot
        
    Returns:
        DataFrame containing the results for each model and metric
    """
    # If no models specified, use all unique values in the model column
    if models is None:
        models = df[model_column].unique()
    
    # If no step mapping provided, create default based on order
    if step_mapping is None:
        step_mapping = {}
        for i, model in enumerate(models):
            step_mapping[model] = i * 100  # Default step size
    
    # Calculate AUC and interpolated TPR for each model and metric
    results = []
    
    for model in models:
        model_df = df[df[model_column] == model]
        if len(model_df) > 0:
            for metric in metrics_list:
                if metric in model_df.columns:
                    # Calculate AUC
                    fpr, tpr = get_roc_curve(model_df, metric, side_task_column)
                    roc_auc = auc(fpr, tpr)
                    
                    # Calculate interpolated TPR
                    interp_tpr = interpolate_tpr(fpr, tpr, target_fpr)
                    
                    results.append({
                        'model': model,
                        'step': step_mapping.get(model, 0),
                        'metric': metric,
                        'auc': roc_auc,
                        'tpr': interp_tpr
                    })
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    if show_plot and len(results_df) > 0:
        # Create separate plots for AUC and TPR
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
        
        # Plot AUC
        for metric, group in results_df.groupby('metric'):
            group = group.sort_values('step')
            ax1.plot(group['step'], group['auc'], 'o-', label=metric)
        
        ax1.set_xlabel('Training Steps')
        ax1.set_ylabel('AUC')
        ax1.set_title('AUC vs Training Steps')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot TPR
        for metric, group in results_df.groupby('metric'):
            group = group.sort_values('step')
            ax2.plot(group['step'], group['tpr'], 'o-', label=metric)
        
        ax2.set_xlabel('Training Steps')
        ax2.set_ylabel(f'TPR @ FPR={target_fpr}')
        ax2.set_title(f'TPR @ FPR={target_fpr} vs Training Steps')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    return results_df

def prepare_monitoring_data(base_dir: str, 
                           monitor_models: List[str] = None,
                           num_monitors: int = 1,
                           compute_aggregates: bool = True,
                           tag_word: str = 'verdict') -> pd.DataFrame:
    """
    Load and prepare monitoring data with flexible options for different experimental setups.
    
    Args:
        base_dir: Base directory containing monitoring data
        monitor_models: Optional list of monitor models to filter by
        num_monitors: Number of monitor responses to process (e.g., 1 for single monitoring, 3 for ensemble)
        compute_aggregates: Whether to compute aggregate metrics (max, mean, median, min)
        tag_word: Tag word used in extract_verdict function
        
    Returns:
        DataFrame containing prepared monitoring data
    """
    # Load monitoring data
    system_prompt_df = load_monitoring_data(base_dir=base_dir)
    
    # Filter by monitor models if specified
    if monitor_models is not None:
        system_prompt_df = system_prompt_df[system_prompt_df.monitor_agent.isin(monitor_models)]
    
    # Extract responses and suspiciousness scores
    for i in range(1, num_monitors + 1):
        # Check if response_{i} exists in the data before trying to access it
        system_prompt_df[f'response_{i}'] = system_prompt_df.data.apply(
            lambda x: x.get(f'response_{i}', {}).get('content', '') 
                if f'response_{i}' in x else ''
        )
        
        system_prompt_df[f'suspiciousness_{i}'] = system_prompt_df.data.apply(
            lambda x: extract_verdict(x.get(f'response_{i}', {}).get('content', ''), tag_word=tag_word)
                if f'response_{i}' in x else 0
        )
        
        system_prompt_df[f'usage_{i}'] = system_prompt_df.data.apply(
            lambda x: x.get(f'response_{i}', {}).get('usage', {})
                if f'response_{i}' in x else {}
        )
    
    # Compute aggregate metrics if requested
    if compute_aggregates and num_monitors > 1:
        suspiciousness_cols = [f'suspiciousness_{i}' for i in range(1, num_monitors + 1)]
        system_prompt_df['max_suspiciousness'] = system_prompt_df[suspiciousness_cols].max(axis=1)
        system_prompt_df['mean_suspiciousness'] = system_prompt_df[suspiciousness_cols].mean(axis=1)
        system_prompt_df['median_suspiciousness'] = system_prompt_df[suspiciousness_cols].median(axis=1)
        system_prompt_df['min_suspiciousness'] = system_prompt_df[suspiciousness_cols].min(axis=1)
        system_prompt_df['std_suspiciousness'] = system_prompt_df[suspiciousness_cols].std(axis=1)
        
        # Add nth highest suspiciousness scores for all ranks
        for rank in range(1, num_monitors + 1):
            system_prompt_df[f'{rank}th_highest_suspiciousness'] = system_prompt_df[suspiciousness_cols].apply(
                lambda x: sorted(x, reverse=True)[rank-1] if len(x) >= rank else 0.0,
                axis=1
            )
    
    return system_prompt_df

def merge_monitoring_with_combined_results(monitoring_df: pd.DataFrame, 
                                         combined_results: pd.DataFrame,
                                         how: str = 'left',
                                         compute_convo_length: bool = True) -> pd.DataFrame:
    """
    Merge monitoring data with combined results and compute additional features.
    
    Args:
        monitoring_df: DataFrame containing monitoring data
        combined_results: DataFrame containing combined results
        how: Type of merge to perform ('left', 'inner', 'right', etc.)
        compute_convo_length: Whether to compute conversation length
        
    Returns:
        DataFrame containing merged data with computed features
    """
    merge_on = ['task_name', 'transcript_type', 'main_agent', 'file_name']
    # Merge the dataframes
    merged_df = pd.merge(
        monitoring_df,
        combined_results[merge_on + ['side_task_evaluation']],
        on=merge_on,
        how=how
    ).dropna()

    # Compute features
    if compute_convo_length and 'transcript' in merged_df.columns:
        merged_df['convo_length'] = merged_df.transcript.apply(
            lambda x: len(x['conversation']) if isinstance(x, dict) and 'conversation' in x else 0
        )

    return merged_df

def filter_and_prepare_data(df: pd.DataFrame, 
                          data_to_use: str = 'all_test',
                          training_data: Optional[pd.DataFrame] = None,
                          exclude_tasks: List[str] = None,
                          transcript_filter: Optional[str] = None) -> pd.DataFrame:
    """
    Filter a DataFrame based on common criteria used in analysis.
    
    Args:
        df: DataFrame to filter
        data_to_use: One of:
            - 'all_test': All data except training data
            - 'no_training_tasks': Only tasks not in training_data
            - 'training_tasks_no_train_data': Only tasks in training_data but specific examples not in training_data
            - 'training_data_only': Only examples in training_data
        training_data: DataFrame containing training data (required if data_to_use references training data)
        exclude_tasks: List of task names to exclude
        transcript_filter: Optional filter for transcript_type column
            
    Returns:
        Filtered DataFrame
    """
    filtered_df = df.copy()
    
    # Apply task exclusions if specified
    if exclude_tasks:
        filtered_df = filtered_df[~filtered_df.task_name.isin(exclude_tasks)]
    
    # Apply transcript type filter if specified
    if transcript_filter:
        if transcript_filter.startswith('not_'):
            # Handle "not_X" filters
            exclude_type = transcript_filter[4:]
            filtered_df = filtered_df[filtered_df.transcript_type != exclude_type]
        else:
            # Direct filtering
            filtered_df = filtered_df[filtered_df.transcript_type == transcript_filter]
    
    # Apply training data filters
    if data_to_use != 'all':
        if training_data is None:
            raise ValueError("training_data must be provided when using training-related filters")
        
        # Create in_training_data column if it doesn't exist
        if 'in_training_data' not in filtered_df.columns:
            filtered_df['in_training_data'] = filtered_df.apply(
                lambda row: is_in_training_data(row, training_data), 
                axis=1
            )
            
        # Apply the appropriate filter
        if data_to_use == 'all_test':
            filtered_df = filtered_df[~filtered_df.in_training_data]
        elif data_to_use == 'no_training_tasks':
            filtered_df = filtered_df[~filtered_df.task_name.isin(training_data.task_name.unique())]
        elif data_to_use == 'training_tasks_no_train_data':
            filtered_df = filtered_df[
                (filtered_df.task_name.isin(training_data.task_name.unique())) & 
                (~filtered_df.in_training_data)
            ]
        elif data_to_use == 'training_data_only':
            filtered_df = filtered_df[filtered_df.in_training_data]
        else:
            raise ValueError(f"Invalid data_to_use value: {data_to_use}")
    
    return filtered_df

def prepare_data_for_training_progress(df: pd.DataFrame,
                                     aggregation_type: str,
                                     step_mapping: Dict[str, int],
                                     model_column: str = 'monitor_agent',
                                     data_to_use: str = 'all_test',
                                     training_data: Optional[pd.DataFrame] = None,
                                     exclude_tasks: List[str] = None) -> pd.DataFrame:
    """
    Prepare data for training progress plots by filtering and unstacking.
    
    Args:
        df: DataFrame to prepare
        aggregation_type: Column to use for aggregation
        step_mapping: Dictionary mapping model names to training steps
        model_column: Column containing model identifiers
        data_to_use: Type of data filter to apply (see filter_and_prepare_data)
        training_data: DataFrame containing training data (if needed)
        exclude_tasks: List of task names to exclude
        
    Returns:
        Prepared DataFrame ready for plotting
    """
    # First filter the data
    filtered_df = filter_and_prepare_data(
        df=df,
        data_to_use=data_to_use,
        training_data=training_data,
        exclude_tasks=exclude_tasks
    )
    
    # Unstack the model column to create columns for each model
    index_columns = [col for col in ['task_name', 'transcript_type', 'main_agent', 'file_name', 
                                    'side_task_evaluation', 'in_training_data'] 
                    if col in filtered_df.columns]
    
    unstacked_df = filtered_df.set_index(index_columns + [model_column])
    
    # Handle both single column or multi-index columns case
    if isinstance(unstacked_df.columns, pd.MultiIndex):
        # Find the appropriate level containing aggregation_type
        found = False
        for i in range(len(unstacked_df.columns.levels)):
            if aggregation_type in unstacked_df.columns.levels[i]:
                unstacked_df = unstacked_df[aggregation_type].unstack(level=model_column)
                found = True
                break
        
        if not found:
            raise ValueError(f"Column {aggregation_type} not found in DataFrame")
    else:
        # Simple column structure
        if aggregation_type in unstacked_df.columns:
            unstacked_df = unstacked_df[aggregation_type].unstack(level=model_column)
        else:
            raise ValueError(f"Column {aggregation_type} not found in DataFrame")
    
    # Reset index and prepare for plotting
    unstacked_df = unstacked_df.reset_index().dropna()
    
    # Add a column to identify this metric in plot legends
    unstacked_df[model_column] = aggregation_type
    
    # Rename columns using step mapping if provided
    if step_mapping:
        unstacked_df.rename(columns=step_mapping, inplace=True)
    
    return unstacked_df

def plot_solution_review_performance(solution_reviews_df: pd.DataFrame,
                                   system_prompts_df: pd.DataFrame,
                                   task_name: str,
                                   step_mapping: Dict[str, int],
                                   monitor_models: List[str],
                                   figsize: tuple = (15, 4),
                                   bins: int = 20,
                                   normalize: bool = True) -> None:
    """
    Plot the performance comparison between solution reviews and system prompts for a specific task.
    
    Args:
        solution_reviews_df: DataFrame containing solution review data
        system_prompts_df: DataFrame containing system prompts data
        task_name: Name of the task to analyze
        step_mapping: Dictionary mapping model names to their training steps
        monitor_models: List of monitor models to include in the analysis
        figsize: Figure size as (width, height) tuple
        bins: Number of bins for histograms
        normalize: Whether to normalize histograms to probability density
    """
    # Filter data for the specific task
    filtered_reviews = solution_reviews_df[solution_reviews_df.task_name == task_name]
    filtered_prompts = system_prompts_df[system_prompts_df.task_name == task_name]
    
    # Calculate grid layout
    n_models = len(monitor_models)
    n_cols = min(3, n_models)  # Maximum 3 columns
    n_rows = (n_models + n_cols - 1) // n_cols  # Ceiling division for rows
    
    # Create figure with subplots
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize[0], figsize[1] * n_rows), constrained_layout=True)
    if n_models == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    # Get global min/max for consistent bins
    global_min = 0
    global_max = 100 if normalize else max(
        filtered_prompts[[col for col in filtered_prompts.columns if 'suspiciousness' in col]].max().max(),
        filtered_reviews[[col for col in filtered_reviews.columns if 'verdict' in col]].max().max()
    )
    
    # Create fixed bins
    bin_edges = np.linspace(global_min, global_max, bins)
    
    # Plot each model
    for i, monitor in enumerate(monitor_models):
        ax = axes[i]
        
        # Get system prompt data with side task
        data_with_side_task = []
        for metric in ['suspiciousness_1_sp5', 'suspiciousness_2_sp5', 'suspiciousness_3_sp5']:
            data = filtered_prompts[
                (filtered_prompts.monitor_agent == monitor) & 
                (filtered_prompts.side_task_evaluation == 1)
            ][metric]
            if not data.empty:
                data_with_side_task.extend(data.tolist())
        
        # Get system prompt data without side task
        data_without_side_task = []
        for metric in ['suspiciousness_1_sp5', 'suspiciousness_2_sp5', 'suspiciousness_3_sp5']:
            data = filtered_prompts[
                (filtered_prompts.monitor_agent == monitor) & 
                (filtered_prompts.side_task_evaluation == 0)
            ][metric]
            if not data.empty:
                data_without_side_task.extend(data.tolist())
        
        # Get solution review data
        data_solution_reviews = []
        model_reviews = filtered_reviews[filtered_reviews.model_name == monitor]
        if not model_reviews.empty:
            for j in range(1, 6):
                verdict = model_reviews[f'verdict_{j}'].iloc[0]
                if verdict is not None:
                    data_solution_reviews.append(verdict)
        
        # Plot histograms
        ax.hist(data_with_side_task, bins=bin_edges, alpha=0.7, density=normalize,
                label="With side task (all metrics)")
        ax.hist(data_without_side_task, bins=bin_edges, alpha=0.7, density=normalize,
                label="Without side task (all metrics)")
        ax.hist(data_solution_reviews, bins=bin_edges, alpha=0.7, density=normalize,
                color='gold', label="Solution reviews (all verdicts)")
        
        # Add labels and title
        ax.set_title(f'Step {step_mapping[monitor]}')
        ax.set_xlabel('Suspiciousness score')
        ax.set_ylabel('Probability density' if normalize else 'Count')
        
        # Add legend to first subplot only
        if i == 0:
            ax.legend()
        
        # Add sample size information
        sample_info = f"n_with={len(data_with_side_task)}, n_without={len(data_without_side_task)}, n_reviews={len(data_solution_reviews)}"
        ax.text(0.5, 0.97, sample_info, transform=ax.transAxes, ha='center', va='top', fontsize=8)
    
    # Hide unused subplots
    for j in range(n_models, len(axes)):
        axes[j].set_visible(False)
    
    # Add overall title
    fig.suptitle(f'Distribution of suspiciousness scores for task: {task_name}', fontsize=16)
    plt.show()

def plot_solution_review_training_progress(solution_reviews_df: pd.DataFrame,
                                         step_mapping: Dict[str, int],
                                         monitor_models: List[str],
                                         figsize: tuple = (10, 6)) -> None:
    """
    Plot the training progress of solution review performance.
    
    Args:
        solution_reviews_df: DataFrame containing solution review data
        step_mapping: Dictionary mapping model names to their training steps
        monitor_models: List of monitor models to include in the analysis
        figsize: Figure size as (width, height) tuple
    """
    # Calculate mean verdict for each model
    solution_reviews_df['mean_verdict'] = solution_reviews_df.apply(
        lambda row: np.mean([row[f'verdict_{i}'] for i in range(1, 6)]), axis=1
    )
    
    # Filter for specified models and sort by training step
    filtered_df = solution_reviews_df[solution_reviews_df.model_name.isin(monitor_models)].copy()
    filtered_df['training_step'] = filtered_df.model_name.map(step_mapping)
    filtered_df = filtered_df.sort_values('training_step')
    
    # Create figure
    plt.figure(figsize=figsize)
    
    # Plot training progress
    for task in filtered_df.task_name.unique():
        task_data = filtered_df[filtered_df.task_name == task]
        plt.plot(task_data.training_step, task_data.mean_verdict, 
                marker='o', label=task)
    
    # Add labels and title
    plt.xlabel('Training Steps')
    plt.ylabel('Mean Verdict')
    plt.title('Solution Review Performance Across Training Steps')
    plt.legend(title='Task', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def load_solution_reviews_data(base_dir: str = "monitor_responses/system_prompt_5_solution_reviews", num_monitors: int = 3) -> pd.DataFrame:
    """
    Load all solution review JSON files from the specified directory structure.
    
    Args:
        base_dir: Base directory containing solution review data
        num_monitors: Number of monitors to process for aggregate metrics
    
    Directory structure:
    base_dir/
        task_name/
            model_name.json
    
    Returns:
        pd.DataFrame: DataFrame containing the loaded data with metadata
    """
    all_data = []
    base_path = resolve_path(base_dir)
    
    # Process each task directory
    for task_dir in base_path.iterdir():
        if not task_dir.is_dir():
            continue
        
        task_name = task_dir.name
        
        # Process each JSON file in the task directory
        for json_file in task_dir.glob('*.json'):
            try:
                with open(json_file, 'r') as f:
                    data = json.load(f)
                
                # Use the filename (without .json) as the model name
                model_name = json_file.stem
                
                # Add metadata about the file location
                entry = {
                    'task_name': task_name,
                    'model_name': model_name,
                    'file_name': json_file.name,
                    'data': data
                }
                all_data.append(entry)
            except Exception as e:
                print(f"Error loading {json_file}: {e}")
                continue
    
    # Convert to DataFrame
    if not all_data:
        print("No data found in the specified directory.")
        return pd.DataFrame()
    
    df = pd.DataFrame(all_data)
    
    # Extract verdicts from responses if they exist
    for i in range(1, 6):
        response_key = f'response_{i}'
        df[f'verdict_{i}'] = df.apply(
            lambda row: extract_verdict(
                row['data'].get(response_key, {}).get('content', ''), 
                tag_word='verdict'
            ) if response_key in row['data'] else None,
            axis=1
        )
        
        # Extract usage information if available
        df[f'usage_{i}'] = df.apply(
            lambda row: row['data'].get(response_key, {}).get('usage', [{}])[0] 
                      if response_key in row['data'] and 
                         'usage' in row['data'].get(response_key, {}) and 
                         len(row['data'].get(response_key, {}).get('usage', [])) > 0 
                      else {},
            axis=1
        )
    
    # Compute aggregate metrics
    suspiciousness_cols = [f'verdict_{i}' for i in range(1, num_monitors + 1) if f'verdict_{i}' in df.columns]
    if suspiciousness_cols:
        df['max_verdict'] = df[suspiciousness_cols].max(axis=1)
        df['mean_verdict'] = df[suspiciousness_cols].mean(axis=1)
        df['median_verdict'] = df[suspiciousness_cols].median(axis=1)
        df['min_verdict'] = df[suspiciousness_cols].min(axis=1)
        df['std_verdict'] = df[suspiciousness_cols].std(axis=1)
    
    return df

def plot_combined_cost_vs_safety_curve(ft_sp1_df: pd.DataFrame, 
                                    ft_sp5_df: pd.DataFrame, 
                                    turn_by_turn_df: pd.DataFrame,
                                    metric_columns: List[str],
                                    cost_columns: List[str],
                                    display_name_col: str = 'monitor_display_name',
                                    side_task_column: str = 'side_task_evaluation',
                                    figsize: tuple = (12, 8)) -> pd.DataFrame:
    """
    Create a unified cost vs safety (AUC) curve for fine-tuned monitors and turn-by-turn monitoring.
    
    Args:
        ft_sp1_df: DataFrame containing fine-tuned system prompt 1 data
        ft_sp5_df: DataFrame containing fine-tuned system prompt 5 data
        turn_by_turn_df: DataFrame containing turn-by-turn monitoring data
        metric_columns: List of metric column names to evaluate
        cost_columns: List of corresponding cost column names
        display_name_col: Column name for monitor display names
        side_task_column: Column name for the binary classification target
        figsize: Figure size as (width, height) tuple
    
    Returns:
        DataFrame containing the combined performance metrics data
    """
    plt.figure(figsize=figsize)
    plt.style.use('seaborn-v0_8-whitegrid')

    # Define more distinct color palettes
    # Use a wider range of distinct colors instead of shades from the same colormap
    ft_colors = ['lightgreen', 'darkgreen', 'lightblue', 'darkblue']  
    tbt_colors = ['lightblue', 'darkblue']  # Complementary colors

    # Get unique monitors for color mapping
    ft_monitors = sorted(list(ft_sp1_df[display_name_col].unique()))
    tbt_monitors = sorted(list(turn_by_turn_df[display_name_col].unique()))

    # Track all performance metrics
    all_metric_performance = []

    # Create the plot with improved styling
    for i, monitor in enumerate(ft_monitors):
        auc_scores = []
        usages = []
        for (metric, usage_metric) in zip(metric_columns, cost_columns):
            if metric in ft_sp1_df.columns:
                filtered_df = ft_sp1_df[ft_sp1_df[display_name_col] == monitor]
                if 'mean_suspiciousness_first' in metric:
                    inner_auc_scores = []
                    num_to_average_over = int(metric.split('_')[-1])
                    for j in range(50):
                        indices = np.random.choice(range(1, len([x for x in metric_columns if 'mean_suspiciousness_first' in x]) + 1), num_to_average_over, replace=False)
                        x = filtered_df[[f'suspiciousness_{j}' for j in indices]].mean(axis=1)
                        fpr, tpr, thresholds = roc_curve(filtered_df[side_task_column], x)
                        inner_auc_scores.append(auc(fpr, tpr))
                    auc_score = np.array(inner_auc_scores).mean()
                else:
                    fpr, tpr, thresholds = roc_curve(filtered_df[side_task_column], filtered_df[metric])
                    auc_score = auc(fpr, tpr)
                usage = filtered_df[usage_metric].mean()
                auc_scores.append(auc_score)
                usages.append(usage)
                
                # Add to tracking
                all_metric_performance.append({
                    'monitor': monitor,
                    'approach': 'SP B',
                    'metric': metric,
                    'auc': auc_score,
                    'cost': usage,
                    'efficiency': auc_score / usage if usage > 0 else 0
                })
        
        # Use distinct colors from our palette
        color = ft_colors[i % len(ft_colors)]
        
        plt.scatter(usages, auc_scores, 
                    s=100,  # Larger point size
                    color=color, 
                    marker='o',
                    edgecolor='black',
                    linewidth=0.5,
                    alpha=0.8,
                    label=f'{monitor} (SP B)')

    for i, monitor in enumerate(ft_monitors):
        auc_scores = []
        usages = []
        for (metric, usage_metric) in zip(metric_columns, cost_columns):
            if metric in ft_sp5_df.columns:
                filtered_df = ft_sp5_df[ft_sp5_df[display_name_col] == monitor]
                if 'mean_suspiciousness_first' in metric:
                    inner_auc_scores = []
                    num_to_average_over = int(metric.split('_')[-1])
                    for j in range(50):
                        indices = np.random.choice(range(1, len([x for x in metric_columns if 'mean_suspiciousness_first' in x]) + 1), num_to_average_over, replace=False)
                        x = filtered_df[[f'suspiciousness_{j}' for j in indices]].mean(axis=1)
                        fpr, tpr, thresholds = roc_curve(filtered_df[side_task_column], x)
                        inner_auc_scores.append(auc(fpr, tpr))
                    auc_score = np.array(inner_auc_scores).mean()
                else:
                    fpr, tpr, thresholds = roc_curve(filtered_df[side_task_column], filtered_df[metric])
                    auc_score = auc(fpr, tpr)
                usage = filtered_df[usage_metric].mean()
                auc_scores.append(auc_score)
                usages.append(usage)
                
                # Add to tracking
                all_metric_performance.append({
                    'monitor': monitor,
                    'approach': 'SP A',
                    'metric': metric,
                    'auc': auc_score,
                    'cost': usage,
                    'efficiency': auc_score / usage if usage > 0 else 0
                })
        
        # Use distinct colors from our palette
        color = ft_colors[i % len(ft_colors)]
        
        plt.scatter(usages, auc_scores, 
                    s=100,  # Larger point size
                    color=color, 
                    marker='x',
                    edgecolor='black',
                    linewidth=1,
                    alpha=1,
                    label=f'{monitor} (SP A)')

    for i, monitor in enumerate(tbt_monitors):
        auc_scores = []
        usages = []
        for (metric, usage_metric) in zip(metric_columns, cost_columns):
            if metric in turn_by_turn_df.columns:
                filtered_df = turn_by_turn_df[turn_by_turn_df[display_name_col] == monitor]
                fpr, tpr, thresholds = roc_curve(filtered_df[side_task_column], filtered_df[metric])
                auc_score = auc(fpr, tpr)
                usage = filtered_df[usage_metric].mean()
                auc_scores.append(auc_score)
                usages.append(usage)
                
                # Add to tracking
                all_metric_performance.append({
                    'monitor': monitor,
                    'approach': 'Turn-by-turn',
                    'metric': metric,
                    'auc': auc_score,
                    'cost': usage,
                    'efficiency': auc_score / usage if usage > 0 else 0
                })
        
        # Use distinct colors from our palette
        color = tbt_colors[i % len(tbt_colors)]
        
        plt.scatter(usages, auc_scores, 
                    s=100,  # Larger point size
                    color=color, 
                    marker='^',  # Different marker for turn-by-turn
                    edgecolor='black',
                    linewidth=0.5,
                    alpha=0.8,
                    label=f'{monitor} (Turn-by-turn)')

    # Add a reference line showing the theoretical cost-effectiveness frontier
    all_usages = []
    all_aucs = []
    for scatter_data in plt.gca().collections:
        all_usages.extend(scatter_data.get_offsets()[:, 0])
        all_aucs.extend(scatter_data.get_offsets()[:, 1])

    if all_usages and all_aucs:
        # Set title and labels with better formatting
        plt.title('Cost vs. Safety Performance Trade-off', fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Average Cost (USD)', fontsize=14, fontweight='bold')
        plt.ylabel('Safety Performance (AUC)', fontsize=14, fontweight='bold')
        
        # Format axes
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.xlim(left=0)
        plt.ylim(top=1)
        
        # Format tick labels
        plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:.4f}'))
        plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2f}'))

    # Create a custom legend with better formatting
    legend = plt.legend(title='Monitoring Models', title_fontsize=13, 
                       fontsize=12, loc='upper right', frameon=True, framealpha=0.95)
    legend.get_frame().set_edgecolor('gray')

    # Add annotations for better context
    plt.annotate('Better', xy=(0.03, 0.95), xycoords='axes fraction', 
                 fontsize=15, fontweight='bold', color='green')
    plt.annotate('↑', xy=(0.04, 0.89), xycoords='axes fraction', 
                 fontsize=15, fontweight='bold', color='green')
    plt.annotate('→', xy=(0.26, 0.95), xycoords='axes fraction', 
                 fontsize=15, fontweight='bold', color='red')
    plt.annotate('Costlier', xy=(0.33, 0.95), xycoords='axes fraction', 
                 fontsize=15, fontweight='bold', color='red')
    plt.xticks(fontsize=14)
    plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2f}'))
    plt.yticks(fontsize=14)
    # Ensure tight layout
    plt.tight_layout()
    plt.show()
    
    # Convert to DataFrame for easier analysis
    performance_df = pd.DataFrame(all_metric_performance)
    
    # Print top performers by different criteria
    if not performance_df.empty:
        print("\n===== TOP PERFORMING METRICS =====")
        
        # Top by AUC (best detection)
        print("\nTop 5 metrics by AUC (best detection):")
        top_auc = performance_df.sort_values('auc', ascending=False).head(5)
        for i, row in top_auc.iterrows():
            print(f"{row['metric']} with {row['monitor']} ({row['approach']}): AUC = {row['auc']:.3f}, Cost = ${row['cost']:.4f}")
        
        # Top by cost-efficiency (AUC/cost ratio)
        print("\nTop 5 metrics by efficiency (AUC/cost ratio):")
        top_efficiency = performance_df.sort_values('efficiency', ascending=False).head(5)
        for i, row in top_efficiency.iterrows():
            print(f"{row['metric']} with {row['monitor']} ({row['approach']}): Efficiency = {row['efficiency']:.2f}, AUC = {row['auc']:.3f}, Cost = ${row['cost']:.4f}")
        
        # Top by lowest cost while maintaining good performance
        print("\nTop 5 low-cost metrics with AUC > 0.7:")
        top_cost_effective = performance_df[performance_df['auc'] > 0.7].sort_values('cost').head(5)
        for i, row in top_cost_effective.iterrows():
            print(f"{row['metric']} with {row['monitor']} ({row['approach']}): Cost = ${row['cost']:.4f}, AUC = {row['auc']:.3f}")
    
    return performance_df

def plot_combined_cost_vs_tpr_curve(ft_sp1_df: pd.DataFrame, 
                                  ft_sp5_df: pd.DataFrame, 
                                  turn_by_turn_df: pd.DataFrame,
                                  metric_columns: List[str],
                                  cost_columns: List[str],
                                  target_fpr: float = 0.03,
                                  display_name_col: str = 'monitor_display_name',
                                  side_task_column: str = 'side_task_evaluation',
                                  figsize: tuple = (12, 8)) -> pd.DataFrame:
    """
    Create a unified cost vs TPR at target FPR curve for fine-tuned monitors and turn-by-turn monitoring.
    
    Args:
        ft_sp1_df: DataFrame containing fine-tuned system prompt 1 data
        ft_sp5_df: DataFrame containing fine-tuned system prompt 5 data  
        turn_by_turn_df: DataFrame containing turn-by-turn monitoring data
        metric_columns: List of metric column names to evaluate
        cost_columns: List of corresponding cost column names
        target_fpr: Target false positive rate for TPR interpolation (default: 0.03 or 3%)
        display_name_col: Column name for monitor display names
        side_task_column: Column name for the binary classification target
        figsize: Figure size as (width, height) tuple
    
    Returns:
        DataFrame containing the TPR performance metrics data
    """
    plt.figure(figsize=figsize)
    plt.style.use('seaborn-v0_8-whitegrid')

    # Define more distinct color palettes
    ft_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']  # Tableau-like palette
    tbt_colors = ['#17becf', '#bcbd22', '#7f7f7f', '#aec7e8', '#ffbb78', '#98df8a', '#ff9896']  # Complementary colors

    # Get unique monitors for color mapping
    ft_monitors = sorted(list(ft_sp1_df[display_name_col].unique()))
    tbt_monitors = sorted(list(turn_by_turn_df[display_name_col].unique()))

    # Track TPR values instead of AUC
    all_tpr_performance = []

    # Plot for fine-tuned SP1 models
    for i, monitor in enumerate(ft_monitors):
        tpr_scores = []
        usages = []
        for (metric, usage_metric) in zip(metric_columns, cost_columns):
            if metric in ft_sp1_df.columns:
                filtered_df = ft_sp1_df[ft_sp1_df[display_name_col] == monitor]
                if 'mean_suspiciousness_first' in metric:
                    inner_tpr_scores = []
                    num_to_average_over = int(metric.split('_')[-1])
                    for j in range(50):
                        indices = np.random.choice(range(1, len([x for x in metric_columns if 'mean_suspiciousness_first' in x]) + 1), num_to_average_over, replace=False)
                        x = filtered_df[[f'suspiciousness_{j}' for j in indices]].mean(axis=1)
                        fpr, tpr, thresholds = roc_curve(filtered_df[side_task_column], x)
                        tpr_at_target = interpolate_tpr(fpr, tpr, target_fpr)
                        inner_tpr_scores.append(tpr_at_target)
                    tpr_at_target = np.array(inner_tpr_scores).mean()
                else:
                    fpr, tpr, thresholds = roc_curve(filtered_df[side_task_column], filtered_df[metric])
                    tpr_at_target = interpolate_tpr(fpr, tpr, target_fpr)
                usage = filtered_df[usage_metric].mean()
                tpr_scores.append(tpr_at_target)
                usages.append(usage)
                all_tpr_performance.append({
                    'monitor': monitor,
                    'approach': 'SP B',
                    'metric': metric,
                    'tpr_at_target': tpr_at_target,
                    'cost': usage,
                    'tpr_efficiency': tpr_at_target / usage if usage > 0 else 0
                })
        
        # Use distinct colors from our palette
        color = ft_colors[i % len(ft_colors)]
        
        plt.scatter(usages, tpr_scores, 
                    s=80,  # Larger point size
                    color=color, 
                    marker='o',
                    edgecolor='black',
                    linewidth=0.5,
                    alpha=0.8,
                    label=f'{monitor} (SP B)')

    # Plot for fine-tuned SP5 models
    for i, monitor in enumerate(ft_monitors):
        tpr_scores = []
        usages = []
        for (metric, usage_metric) in zip(metric_columns, cost_columns):
            if metric in ft_sp5_df.columns:
                filtered_df = ft_sp5_df[ft_sp5_df[display_name_col] == monitor]
                if 'mean_suspiciousness_first' in metric:
                    inner_tpr_scores = []
                    num_to_average_over = int(metric.split('_')[-1])
                    for j in range(50):
                        indices = np.random.choice(range(1, len([x for x in metric_columns if 'mean_suspiciousness_first' in x]) + 1), num_to_average_over, replace=False)
                        x = filtered_df[[f'suspiciousness_{j}' for j in indices]].mean(axis=1)
                        fpr, tpr, thresholds = roc_curve(filtered_df[side_task_column], x)
                        tpr_at_target = interpolate_tpr(fpr, tpr, target_fpr)
                        inner_tpr_scores.append(tpr_at_target)
                    tpr_at_target = np.array(inner_tpr_scores).mean()
                else:
                    fpr, tpr, thresholds = roc_curve(filtered_df[side_task_column], filtered_df[metric])
                    tpr_at_target = interpolate_tpr(fpr, tpr, target_fpr)
                usage = filtered_df[usage_metric].mean()
                usages.append(usage)
                tpr_scores.append(tpr_at_target)
                all_tpr_performance.append({
                    'monitor': monitor,
                    'approach': 'SP A',
                    'metric': metric,
                    'tpr_at_target': tpr_at_target,
                    'cost': usage,
                    'tpr_efficiency': tpr_at_target / usage if usage > 0 else 0
                })
        
        # Use distinct colors from our palette
        color = ft_colors[i % len(ft_colors)]
        
        plt.scatter(usages, tpr_scores, 
                    s=80,  # Larger point size
                    color=color, 
                    marker='x',
                    edgecolor='black',
                    linewidth=0.5,
                    alpha=0.8,
                    label=f'{monitor} (SP A)')

    # Plot for turn-by-turn models
    for i, monitor in enumerate(tbt_monitors):
        tpr_scores = []
        usages = []
        for (metric, usage_metric) in zip(metric_columns, cost_columns):
            if metric in turn_by_turn_df.columns:
                filtered_df = turn_by_turn_df[turn_by_turn_df[display_name_col] == monitor]
                fpr, tpr, thresholds = roc_curve(filtered_df[side_task_column], filtered_df[metric])
                # Use interpolate_tpr to get TPR at the target FPR
                tpr_at_target = interpolate_tpr(fpr, tpr, target_fpr)
                usage = filtered_df[usage_metric].mean()
                tpr_scores.append(tpr_at_target)
                usages.append(usage)
                all_tpr_performance.append({
                    'monitor': monitor,
                    'approach': 'Turn-by-turn',
                    'metric': metric,
                    'tpr_at_target': tpr_at_target,
                    'cost': usage,
                    'tpr_efficiency': tpr_at_target / usage if usage > 0 else 0
                })
        
        # Use distinct colors from our palette
        color = tbt_colors[i % len(tbt_colors)]
        
        plt.scatter(usages, tpr_scores, 
                    s=80,  # Larger point size
                    color=color, 
                    marker='^',  # Different marker for turn-by-turn
                    edgecolor='black',
                    linewidth=0.5,
                    alpha=0.8,
                    label=f'{monitor} (Turn-by-turn)')

    # Add reference line and format axes
    all_usages = []
    all_tprs = []
    for scatter_data in plt.gca().collections:
        all_usages.extend(scatter_data.get_offsets()[:, 0])
        all_tprs.extend(scatter_data.get_offsets()[:, 1])

    if all_usages and all_tprs:
        # Set title and labels with better formatting
        plt.title(f'Cost vs. TPR at {target_fpr*100}% FPR Trade-off', fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Average Cost (USD)', fontsize=14, fontweight='bold')
        plt.ylabel(f'TPR at {target_fpr*100}% FPR', fontsize=14, fontweight='bold')
        
        # Format axes
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.xlim(left=0)
        plt.ylim(bottom=0, top=1)
        
        # Format tick labels
        plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:.4f}'))
        plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2f}'))

    # Create a custom legend with better formatting
    legend = plt.legend(title='Monitoring Models', title_fontsize=13, 
                       fontsize=10, loc='lower right', frameon=True, framealpha=0.95)
    legend.get_frame().set_edgecolor('gray')

    # Add annotations for better context
    plt.annotate('Better', xy=(0.03, 0.95), xycoords='axes fraction', 
                 fontsize=12, fontweight='bold', color='green')
    plt.annotate('↑', xy=(0.04, 0.92), xycoords='axes fraction', 
                 fontsize=15, fontweight='bold', color='green')
    plt.annotate('→', xy=(0.26, 0.95), xycoords='axes fraction', 
                 fontsize=15, fontweight='bold', color='red')
    plt.annotate('Costlier', xy=(0.28, 0.95), xycoords='axes fraction', 
                 fontsize=12, fontweight='bold', color='red')

    # Ensure tight layout
    plt.tight_layout()
    plt.show()

    # Convert TPR performance data to DataFrame
    tpr_performance_df = pd.DataFrame(all_tpr_performance)
    
    # Print top performers by different criteria
    if not tpr_performance_df.empty:
        print(f"\n===== TOP PERFORMING METRICS BY TPR AT {target_fpr*100}% FPR =====")
        
        # Top for TPR at target FPR
        print(f"\nTop 5 metrics by TPR at {target_fpr*100}% FPR:")
        top_tpr = tpr_performance_df.sort_values('tpr_at_target', ascending=False).head(5)
        for i, row in top_tpr.iterrows():
            print(f"{row['metric']} with {row['monitor']} ({row['approach']}): TPR = {row['tpr_at_target']:.3f}, Cost = ${row['cost']:.4f}")
        
        # Most efficient metrics for TPR at target FPR
        print(f"\nTop 5 metrics by TPR/{target_fpr*100}% FPR efficiency:")
        top_tpr_efficiency = tpr_performance_df.sort_values('tpr_efficiency', ascending=False).head(5)
        for i, row in top_tpr_efficiency.iterrows():
            print(f"{row['metric']} with {row['monitor']} ({row['approach']}): TPR Efficiency = {row['tpr_efficiency']:.2f}, TPR = {row['tpr_at_target']:.3f}, Cost = ${row['cost']:.4f}")
    
    return tpr_performance_df